In [45]:
#This program reads dataset tables from NCBI Genome searches and sorts extremophilic candidates and control candidates
##This program will also analyze sequence length and contig N50 values
###This program will compile the selection of alicyclobacillaceae control candidates

In [46]:
import pandas as pd

#loading the NCBI genome search result table
alicyclobacillaceaeTable = pd.read_csv("/content/drive/MyDrive/Capstone/Data/Acidos/Pre-Processing/AlicyclobacillaceaeGenomes.tsv", sep='\t')
print(alicyclobacillaceaeTable.columns)
print(alicyclobacillaceaeTable.head())
print("\n", "There are ", len(alicyclobacillaceaeTable), "species genomes in the NCBI search")

Index(['Assembly Accession', 'Assembly Paired Assembly Accession',
       'Organism Name', 'Organism Infraspecific Names Breed',
       'Organism Infraspecific Names Strain',
       'Organism Infraspecific Names Cultivar',
       'Organism Infraspecific Names Ecotype',
       'Organism Infraspecific Names Isolate',
       'Organism Infraspecific Names Sex', 'Annotation Name',
       'Assembly Stats Total Sequence Length', 'Assembly Stats Contig N50',
       'Assembly Stats Number of Scaffolds', 'Assembly Stats GC Percent',
       'CheckM completeness', 'CheckM contamination'],
      dtype='object')
  Assembly Accession Assembly Paired Assembly Accession  \
0    GCA_000024285.1                    GCF_000024285.1   
1    GCF_000024285.1                    GCA_000024285.1   
2    GCA_022674245.1                    GCF_022674245.1   
3    GCF_022674245.1                    GCA_022674245.1   
4    GCA_026651875.1                    GCF_026651875.1   

                                       

In [47]:
#Read in the Alicyclobacillaceae acidophile csv
acidophiles = pd.read_csv("/content/drive/MyDrive/Capstone/Data/Acidos/Post-Processing/alicyclobacillusAcidos.csv", header=None)
print(acidophiles.columns.tolist())
print(acidophiles.head())
print(f"\n{len(acidophiles)} rows loaded")


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
                                 0            1         2           3   \
0        Acidibacillus ferrooxidans        ITV01  Bacteria  Firmicutes   
1       Acidibacillus sulfuroxidans         Y002  Bacteria  Firmicutes   
2      Alicyclobacillus acidiphilus  NBRC 100859  Bacteria  Firmicutes   
3   Alicyclobacillus acidocaldarius       Tc-4-1  Bacteria  Firmicutes   
4  Alicyclobacillus acidoterrestris   ATCC 49025  Bacteria  Firmicutes   

        4                5    6       7   \
0  GenBank  GCA_001516615.1  yes   96.98   
1  GenBank  GCA_003144315.1  yes   98.08   
2  GenBank  GCA_001544355.1  yes   99.21   
3  GenBank  GCA_000219875.1  yes  100.00   
4  GenBank  GCA_000444055.1  yes   98.82   

                                                  8   \
0  ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/001...   
1  ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/003...   
2  ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/001...   
3  ftp://ftp.ncbi.nlm.nih.gov/g

In [48]:
#extract the accession numbers for all acidophiles
acidophileAccessions = acidophiles[5].dropna().str.strip().tolist()

#verify previously selected Alicyclobacillaceae acidophiles are in the NCBI genome search results
#The number of acidophile and non-acidophile genomes in the set will be double what
  ##it should because I accidentally have duplicates for GCA and GCF accessions.
  ###Basically the NCBI list has everything duplicated on accident :(

#Execute matching function using accession numbers instead of names
alicyclobacillaceaeTable["Assignment"] = alicyclobacillaceaeTable.apply(
    lambda row: "Acidophile" if (
        row["Assembly Accession"] in acidophileAccessions or
        row["Assembly Paired Assembly Accession"] in acidophileAccessions
    ) else "Non-Acidophile",
    axis=1
)

#Display organism assignments
print("Below are the number of genomes available in each group", "\n")
print(alicyclobacillaceaeTable["Assignment"].value_counts())

Below are the number of genomes available in each group 

Assignment
Non-Acidophile    189
Acidophile         38
Name: count, dtype: int64


In [49]:
#Compiling all the acidophile GCF accessions

def get_GCFs(gca_accession, table):
    row = table[table["Assembly Accession"] == gca_accession]
    paired = row["Assembly Paired Assembly Accession"].values[0]
    if str(paired).startswith("GCF"):
        return paired
    else:
        return gca_accession  # no GCF pair exists, keep original GCA

acidophileAccessions = [
    get_GCFs(acc, alicyclobacillaceaeTable) for acc in acidophileAccessions]

print(f"{len(acidophileAccessions)} GCF acidophile accessions")
print(acidophileAccessions)


19 GCF acidophile accessions
['GCF_001516615.1', 'GCF_003144315.1', 'GCF_001544355.1', 'GCF_000219875.1', 'GCF_000444055.1', 'GCF_000429525.1', 'GCF_001399675.1', 'GCF_000430585.1', 'GCF_900107035.1', 'GCF_001552655.1', 'GCF_900116805.1', 'GCF_001570745.1', 'GCF_000472905.1', 'GCF_004366795.1', 'GCF_001552255.1', 'GCF_001447355.1', 'GCF_900156755.1', 'GCF_002804065.1', 'GCF_000092905.1']


In [71]:
#write the acidophileAccessions as a txt so they can be correctly retrieved from GenBank later
pd.Series(acidophileAccessions).to_csv("/content/drive/MyDrive/Capstone/Data/Acidos/Post-Processing/acidophileAccessions.txt",index=False, header=False)
print(f"Wrote {len(acidophileAccessions)} accessions to .txt")

Wrote 19 accessions to .txt


In [72]:
#Filtering out the acidophiles to create the mesophile/control group to compare
alicyclobacillaceaeMesos = alicyclobacillaceaeTable[alicyclobacillaceaeTable["Assignment"] == "Non-Acidophile"].copy()

#Filtering for genome quality: completeness >= 95% and contamination < 5%
alicyclobacillaceaeMesos = alicyclobacillaceaeMesos[
    (alicyclobacillaceaeMesos["CheckM completeness"] >= 95) &
    (alicyclobacillaceaeMesos["CheckM contamination"] <= 5)].copy()

#Keeping only genomes with a GCF accession (excludes GCA and NaN)
alicyclobacillaceaeMesos = alicyclobacillaceaeMesos[
    alicyclobacillaceaeMesos["Assembly Accession"].str.startswith("GCF", na=False)].copy()

#Dedupe by organism name (all remaining rows are already GCF)
alicyclobacillaceaeMesos = alicyclobacillaceaeMesos.drop_duplicates(subset=["Organism Name"]).copy()

#Selecting mesophiles that are in the same genera as the acidophiles
acidophileGenera = alicyclobacillaceaeTable[
    alicyclobacillaceaeTable["Assignment"] == "Acidophile"]["Organism Name"].str.split().str[0].str.lower().unique()

alicyclobacillaceaeMesos["Genus"] = alicyclobacillaceaeMesos["Organism Name"].str.split().str[0].str.lower()
matchedMesos = alicyclobacillaceaeMesos[alicyclobacillaceaeMesos["Genus"].isin(acidophileGenera)]

print(acidophileGenera, "\n")
print("There are ", len(alicyclobacillaceaeMesos), " mesophiles that match the quality requirements")
print("There are ", len(matchedMesos), " mesophiles that match the acidophile genera and quality requirements")

['kyrpidia' 'paenalicyclobacillus' 'alicyclobacillus' 'sulfoacidibacillus'
 'ferroacidibacillus'] 

There are  22  mesophiles that match the quality requirements
There are  19  mesophiles that match the acidophile genera and quality requirements


In [73]:
#compiling the mesophile GCF accessions and write to txt file
alicyclobacillaceaeMesos["Assembly Accession"].to_csv("/content/drive/MyDrive/Capstone/Data/Acidos/Post-Processing/alicyclomesoAccessions.txt", index=False, header=False)
print(f"Wrote {len(alicyclobacillaceaeMesos)} accessions to .txt")

Wrote 22 accessions to .txt


In [74]:
#Combining the acido and meso accessions into one list
selectedAccessions = list(acidophileAccessions) + alicyclobacillaceaeMesos["Assembly Accession"].tolist()

#making a copy of the ncbi results that only includes instances that we are going to pull genomes for
alicyclobacillaceaeFiltered = alicyclobacillaceaeTable[
    alicyclobacillaceaeTable["Assembly Accession"].isin(selectedAccessions)].copy()

print(f"Selected {len(alicyclobacillaceaeFiltered)} genomes total ({len(acidophileAccessions)} acidophiles + {len(alicyclobacillaceaeMesos)} mesophiles)")

Selected 41 genomes total (19 acidophiles + 22 mesophiles)


In [75]:
#writing the compiled acido and meso alicyclobacillaceaeTable for EDA
alicyclobacillaceaeFiltered.to_csv("/content/drive/MyDrive/Capstone/Data/Acidos/Post-Processing/alicyclobacillaceaeGenomes.csv", index=False)

print(f"Wrote {len(alicyclobacillaceaeFiltered)} rows to alicyclobacillaceaeFiltered.csv")

Wrote 41 rows to alicyclobacillaceaeFiltered.csv
